In [320]:
# Cell: fetch_spatis.ipynb / fetch_spatis.py
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Overpass query for Berlin bounding box (conservative)
query = """
[out:json];
(
  node[shop=convenience](52.4,13.2,52.7,13.6);
  node[shop=kiosk](52.4,13.2,52.7,13.6);
);
out body;
"""

url = "https://overpass-api.de/api/interpreter"
resp = requests.post(url, data=query, timeout=180)
resp.raise_for_status()
data = resp.json()

elements = data.get("elements", [])
df = pd.json_normalize(elements)

# ensure lat/lon exist
df = df[df['lat'].notna() & df['lon'].notna()]

# create GeoDataFrame
gdf = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df['lon'].astype(float), df['lat'].astype(float))],
    crs="EPSG:4326"
)

# minimal cleanup/rename to match your schema
gdf = gdf.rename(columns={
    'tags.name': 'name',
    'tags.brand': 'brand',
    'tags.operator': 'operator',
    'tags.opening_hours': 'opening_hours',
    'tags.phone': 'phone',
    'tags.website': 'website',
    'tags.source': 'source',
    'lat': 'latitude',
    'lon': 'longitude',
    'type': 'osm_type',
    'id': 'osm_id'
})

# assign to expected variable name
spatis_gdf = gdf.copy()
print("spatis_gdf created with", len(spatis_gdf), "records")



spatis_gdf created with 1607 records


In [232]:

# ---------------------------
# Existing code fetches SPATIS from OSM
# ---------------------------
# (Your original fetch_spatis code here)
# ...
spatis_gdf = gdf.copy()
print("spatis_gdf created with", len(spatis_gdf), "records")

# ---------------------------
# Step 1: Add district_name and district_id
# ---------------------------
# Use OSM suburb as district name
spatis_gdf['district_name'] = spatis_gdf['tags.addr:suburb']

# Map official Berlin district codes
district_mapping = {
    'Mitte': '11001001',
    'Friedrichshain-Kreuzberg': '11002002',
    'Pankow': '11003003',
    'Charlottenburg-Wilmersdorf': '11004004',
    'Spandau': '11005005',
    'Steglitz-Zehlendorf': '11006006',
    'Tempelhof-Schöneberg': '11007007',
    'Neukölln': '11008008',
    'Treptow-Köpenick': '11009009',
    'Marzahn-Hellersdorf': '11010010',
    'Lichtenberg': '11011011',
    'Reinickendorf': '11012012'
}

spatis_gdf['district_id'] = spatis_gdf['district_name'].map(district_mapping)

# Optional: check unmapped districts
unmapped = spatis_gdf[spatis_gdf['district_id'].isna()]['district_name'].unique()
if len(unmapped) > 0:
    print("⚠️ Unmapped districts found:", unmapped)

# ---------------------------
# Step 2: Save updated SPATIS GeoJSON
# ---------------------------
import os
os.makedirs("spatis", exist_ok=True)
spatis_fp = "spatis/spatis_data.geojson"
spatis_gdf.to_file(spatis_fp, driver="GeoJSON")
print(f"SPATIS file saved with district info: {spatis_fp}")




spatis_gdf created with 1607 records
⚠️ Unmapped districts found: ['Halensee' nan 'Kaulsdorf' 'Kreuzberg' 'Moabit' 'Friedrichshain'
 'Köpenick' 'Wilmersdorf' 'Schöneberg' 'Gesundbrunnen' 'Lichterfelde'
 'Prenzlauer Berg' 'Friedenau' 'Wedding' 'Baumschulenweg' 'Weißensee'
 'Charlottenburg' 'Oberschöneweide' 'Tempelhof' 'Rudow' 'Zehlendorf'
 'Lübars' 'Steglitz' 'Haselhorst' 'Siemensstadt' 'Alt-Treptow' 'Lankwitz'
 'Fennpfuhl' 'Westend' 'Tegel' 'Adlershof' 'Schildow' 'Heinersdorf'
 'Niederschönhausen' 'Karlshorst' 'Mühlenbeck' 'Schlachtensee' 'Grünau'
 'Rummelsburg' 'Schwanebeck' 'Wittenau' 'Hakenfelde' 'Mariendorf'
 'Marzahn' 'Alt-Hohenschönhausen' 'Zepernick' 'Wilhelmsruh' 'Biesdorf']
SPATIS file saved with district info: spatis/spatis_data.geojson


In [233]:
import requests
import pandas as pd
import os

# Corrected Overpass query for Spätis/corner stores
query = """
[out:json];
(
node[shop=convenience](52.4,13.2,52.7,13.6);
  node[shop=kiosk](52.4,13.2,52.7,13.6);
);
out;
"""

url = "https://overpass-api.de/api/interpreter"

response = requests.post(url, data=query)

if response.status_code == 200:
    try:
        data = response.json()
        elements = data.get("elements", [])
        df = pd.json_normalize(elements)

        # Save CSV to Documents
        folder = "/Users/harrisongoodman/Documents"
        os.makedirs(folder, exist_ok=True)
        file_path = os.path.join(folder, "spatis.csv")
        df.to_csv(file_path, index=False)
        print(f"CSV saved successfully to: {file_path}")

    except ValueError:
        print("Response is not in JSON format:")
        print(response.text[:200])
else:
    print(f"Request failed with status code: {response.status_code}")


CSV saved successfully to: /Users/harrisongoodman/Documents/spatis.csv


In [234]:
import os

# Ensure output folder exists
spatis_folder = "spatis"
os.makedirs(spatis_folder, exist_ok=True)

# Save the SPATIS GeoDataFrame to GeoJSON
spatis_fp = os.path.join(spatis_folder, "spatis_data.geojson")
spatis_gdf.to_file(spatis_fp, driver="GeoJSON")

print(f"Saved SPATIS file to: {spatis_fp}")


Saved SPATIS file to: spatis/spatis_data.geojson


In [213]:
spatis_fp = "spatis/spatis_data.geojson"




In [235]:
# Suppose your GeoDataFrame is called spatis_gdf
column_names = spatis_gdf.columns.to_list()
# Display all column names
print(column_names)




['osm_type', 'osm_id', 'latitude', 'longitude', 'tags.addr:city', 'tags.addr:country', 'tags.addr:housenumber', 'tags.addr:postcode', 'tags.addr:street', 'tags.addr:suburb', 'tags.amenity', 'tags.check_date:opening_hours', 'tags.compressed_air', 'tags.fuel:adblue', 'tags.fuel:biodiesel', 'tags.fuel:diesel', 'tags.fuel:e10', 'tags.fuel:octane_95', 'tags.fuel:octane_98', 'name', 'opening_hours', 'operator', 'tags.shop', 'tags.wheelchair', 'brand', 'tags.brand:wikidata', 'tags.brand:wikipedia', 'tags.fuel:GTL_diesel', 'tags.fuel:biogas', 'tags.fuel:cng', 'tags.fuel:lpg', 'tags.fuel:octane_102', 'tags.surveillance', 'website', 'tags.check_date', 'tags.dog', 'tags.email', 'tags.fax', 'phone', 'tags.start_date', 'tags.indoor_seating', 'tags.organic', 'tags.outdoor_seating', 'tags.smoking', 'tags.opening_hours:signed', 'tags.diet:halal', 'tags.level', 'tags.payment:credit_cards', 'tags.payment:debit_cards', 'tags.payment:apple_pay', 'tags.payment:cards', 'tags.payment:cash', 'tags.payment:goo

In [236]:
# Explore all columns in the Spätis GeoDataFrame
spatis_gdf.describe(include="all").T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
osm_type,1607,1,node,1607,NaN,NaN,NaN,NaN,NaN,NaN,NaN
osm_id,1607.0,NaN,NaN,NaN,5371354629.167393,3786370045.437481,26867411.0,1983338789.5,4520970753.0,8332127068.0,13306244164.0
latitude,1607.0,NaN,NaN,NaN,52.512142,0.040235,52.401392,52.488105,52.510031,52.540426,52.689244
longitude,1607.0,NaN,NaN,NaN,13.396389,0.068748,13.200417,13.347769,13.402107,13.439757,13.592819
tags.addr:city,686,6,Berlin,677,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
tags.public_transport,1,1,service_center,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tags.ticket,1,1,public_transport,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
geometry,1607,1605,POINT (13.3129938 52.5017546),2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
district_name,475,53,Prenzlauer Berg,59,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [237]:
# Count missing values in each column
missing_count = spatis_gdf.isna().sum().sort_values(ascending=False)

# Show columns that have more than 200 missing values
print(missing_count[missing_count > 200])


tags.post_box                 1606
tags.post_office:post_bank    1606
tags.cash_in                  1606
tags.old_addr:housenumber     1606
tags.old_addr:street          1606
                              ... 
tags.addr:postcode             900
opening_hours                  888
tags.addr:housenumber          812
tags.addr:street               778
tags.wheelchair                604
Length: 243, dtype: int64


In [238]:
# check unique values in 'barand' column
spatis_gdf['brand'].value_counts()

brand
REWE To Go                                  16
ServiceStore DB                             15
DHL                                          4
Yorma's                                      2
Total                                        2
Spar                                         2
DPD                                          1
JET                                          1
Shell Shop                                   1
Shell                                        1
Weltladen                                    1
Aral                                         1
Deutsche Post                                1
EDEKA                                        1
Elan                                         1
Lycamobile                                   1
Deutsche Post;DHL;Postbank;Western Union     1
Agip                                         1
Hermes                                       1
Edeka                                        1
TotalEnergies                                1
Name: c

In [239]:
#expland all columns to see more details
pd.set_option('display.max_columns', None)
print(spatis_gd.head(3))

                                name brand       operator  \
element id                                                  
node    63253672           Späti Joe   NaN            NaN   
        285096511  Späti am Comenius   NaN  H.Y. Soysüren   
        427670051              Späti   NaN            NaN   

                                addr:street addr:housenumber addr:postcode  \
element id                                                                   
node    63253672   Joachim-Friedrich-Straße               39         10711   
        285096511            Gubener Straße               43         10243   
        427670051                       NaN              NaN           NaN   

                      addr:suburb addr:city addr:country phone email website  \
element id                                                                     
node    63253672         Halensee    Berlin           DE   NaN   NaN     NaN   
        285096511  Friedrichshain    Berlin           DE   NaN 

In [240]:
# Selected Columns & Add Coordinates

In [241]:
import geopandas as gpd

# Example: Load your GeoDataFrame if not already loaded
# spatis_gdf = gpd.read_file("your_file.geojson")  # or .shp, etc.

# Ensure it's a GeoDataFrame
if not isinstance(spatis_gdf, gpd.GeoDataFrame):
    spatis_gdf = gpd.GeoDataFrame(spatis_gdf, geometry='geometry')

# Check the current CRS
print("Current CRS:", spatis_gdf.crs)

# Set CRS to WGS84 (lat/lon) if not already
spatis_gdf = spatis_gdf.to_crs(epsg=4326)

# Optional: Ensure geometry type is Point (needed for lat/lon extraction)
spatis_gdf = spatis_gdf[spatis_gdf.geometry.type == "Point"]

# Now you can extract lat/lon
spatis_gdf['latitude'] = spatis_gdf.geometry.y
spatis_gdf['longitude'] = spatis_gdf.geometry.x

# Quick check
print(spatis_gdf[['latitude', 'longitude']].head())


Current CRS: EPSG:4326
    latitude  longitude
0  52.501974  13.294496
1  52.508370  13.280947
2  52.499322  13.296118
3  52.511047  13.462698
4  52.509209  13.587500


In [242]:
spatis_gdf['geometry'] = spatis_gdf['geometry'].apply(lambda geom: geom if geom.geom_type == 'Point' else geom.representative_point())
# Extract latitude and longitude
spatis_gdf["latitude"] = spatis_gdf.geometry.y
spatis_gdf["longitude"] = spatis_gdf.geometry.x
spatis_gdf


,osm_type,osm_id,latitude,longitude,tags.addr:city,tags.addr:country,tags.addr:housenumber,tags.addr:postcode,tags.addr:street,tags.addr:suburb,tags.amenity,tags.check_date:opening_hours,tags.compressed_air,tags.fuel:adblue,tags.fuel:biodiesel,tags.fuel:diesel,tags.fuel:e10,tags.fuel:octane_95,tags.fuel:octane_98,name,opening_hours,operator,tags.shop,tags.wheelchair,brand,tags.brand:wikidata,tags.brand:wikipedia,tags.fuel:GTL_diesel,tags.fuel:biogas,tags.fuel:cng,tags.fuel:lpg,tags.fuel:octane_102,tags.surveillance,website,tags.check_date,tags.dog,tags.email,tags.fax,phone,tags.start_date,tags.indoor_seating,tags.organic,tags.outdoor_seating,tags.smoking,tags.opening_hours:signed,tags.diet:halal,tags.level,tags.payment:credit_cards,tags.payment:debit_cards,tags.payment:apple_pay,tags.payment:cards,tags.payment:cash,tags.payment:google_pay,tags.payment:paypal,tags.drink:club-mate,tags.entrance,tags.FIXME,tags.created_by,tags.name:signed,tags.toilets:wheelchair,tags.wheelchair:description,tags.noname,tags.post_office,tags.post_office:service_provider,tags.lottery,tags.post_office:brand,tags.post_office:ref,tags.origin,tags.alt_name,tags.cuisine,tags.ref:vatin,tags.name:de,tags.name:en,tags.addr:floor,tags.air_conditioning,tags.contact:instagram,tags.contact:website,tags.internet_access,tags.stroller,tags.toilets,tags.access:covid19,tags.delivery:covid19,tags.post_office:type,tags.contact:fax,tags.contact:phone,tags.name:ru,tags.opening_hours:covid19,tags.takeaway:covid19,tags.internet_access:fee,tags.atm,tags.atm:operator,tags.note:de,tags.ref:Hermes,tags.description,tags.coffee,tags.tobacco,tags.post_office:brand:wikidata,tags.note,tags.operator:wikidata,tags.ref,tags.currency:EUR,source,tags.diet:gluten_free,tags.leisure,tags.cafe,tags.payment:coins,tags.vending:sweets,tags.vending:toys,tags.service:phone,tags.type,tags.diet:kosher,tags.diet:vegetarian,tags.post_box,tags.operator:wikipedia,tags.changing_table,tags.payment:mastercard,tags.payment:visa,tags.second_hand,tags.toilets:menstrual_products,tags.takeaway,tags.toilets:access,tags.parcel_pickup,tags.drink:coffee,tags.sells:tobacco,tags.name:zh,tags.addr:housename,tags.fixme,tags.contact:mobile,tags.dry_cleaning,tags.ice_cream,tags.drink:beer,tags.tickets:public_transport,tags.diet:vegan,tags.branch,tags.bulk_purchase,tags.diet:organic,tags.frozen_yogurt,tags.reusable_packaging:offer,tags.contact:email,tags.lgbtq,tags.post_office:letter,tags.post_office:parcel_to,tags.addr:place,tags.post_office:parcel_from,tags.post_office:parcel_pickup,tags.ref:deutsche_post,tags.tourism,tags.fair_trade,tags.payment:contactless,tags.payment:maestro,tags.payment:v_pay,tags.payment:app,tags.mapillary,tags.opening_date,tags.contact:facebook,tags.post_office:operator,tags.cash_in,tags.survey:date,tags.post_office:letter_from,tags.post_office:packaging,tags.post_office:stamps,tags.payment:girocard,tags.description:en,tags.old_name,tags.old_addr:housenumber,tags.old_addr:street,tags.reusable_packaging:accept,tags.wikidata,tags.wikipedia,tags.zero_waste,tags.not:brand:wikidata,tags.post_office:post_bank,tags.building,tags.roof:material,tags.roof:shape,tags.vending,tags.layer,tags.disused:shop,tags.service:copy,tags.service:fax,tags.service:scan,tags.self_checkout,tags.payment:american_express,tags.payment:notes,tags.payment:telephone_cards,tags.payment:credit_cards:min_payment,tags.loc_name,tags.delivery,tags.bicycle,tags.craft,tags.indoor:level,tags.name:bg,tags.name:fa,tags.ramp:wheelchair,tags.toilets:charge,tags.smoking:outside,tags.description:name,tags.name:ar,tags.post_office:service_provider:wikidata,tags.wheelchair:description:de,tags.wheelchair:description:en,tags.payment:account_cards,tags.payment:alipay,tags.payment:bancomat,tags.payment:blik,tags.payment:cheque,tags.payment:clipper,tags.payment:cryptocurrencies,tags.payment:diners_club,tags.payment:discover_card,tags.payment:dkv,tags.payment:electronic_purses,tags.payment:ep_easycard,tags.payment:ep_geldkarte,tags.payment:ep_ipass

In [243]:
# Convert non-point geometries (e.g., polygons) into representative points
spatis_gdf['geometry'] = spatis_gdf['geometry'].apply(
    lambda geom: geom if geom.geom_type == 'Point' else geom.representative_point()
)

# Extract latitude and longitude
spatis_gdf["latitude"] = spatis_gdf.geometry.y
spatis_gdf["longitude"] = spatis_gdf.geometry.x

# Display the updated GeoDataFrame
spatis_gdf


,osm_type,osm_id,latitude,longitude,tags.addr:city,tags.addr:country,tags.addr:housenumber,tags.addr:postcode,tags.addr:street,tags.addr:suburb,tags.amenity,tags.check_date:opening_hours,tags.compressed_air,tags.fuel:adblue,tags.fuel:biodiesel,tags.fuel:diesel,tags.fuel:e10,tags.fuel:octane_95,tags.fuel:octane_98,name,opening_hours,operator,tags.shop,tags.wheelchair,brand,tags.brand:wikidata,tags.brand:wikipedia,tags.fuel:GTL_diesel,tags.fuel:biogas,tags.fuel:cng,tags.fuel:lpg,tags.fuel:octane_102,tags.surveillance,website,tags.check_date,tags.dog,tags.email,tags.fax,phone,tags.start_date,tags.indoor_seating,tags.organic,tags.outdoor_seating,tags.smoking,tags.opening_hours:signed,tags.diet:halal,tags.level,tags.payment:credit_cards,tags.payment:debit_cards,tags.payment:apple_pay,tags.payment:cards,tags.payment:cash,tags.payment:google_pay,tags.payment:paypal,tags.drink:club-mate,tags.entrance,tags.FIXME,tags.created_by,tags.name:signed,tags.toilets:wheelchair,tags.wheelchair:description,tags.noname,tags.post_office,tags.post_office:service_provider,tags.lottery,tags.post_office:brand,tags.post_office:ref,tags.origin,tags.alt_name,tags.cuisine,tags.ref:vatin,tags.name:de,tags.name:en,tags.addr:floor,tags.air_conditioning,tags.contact:instagram,tags.contact:website,tags.internet_access,tags.stroller,tags.toilets,tags.access:covid19,tags.delivery:covid19,tags.post_office:type,tags.contact:fax,tags.contact:phone,tags.name:ru,tags.opening_hours:covid19,tags.takeaway:covid19,tags.internet_access:fee,tags.atm,tags.atm:operator,tags.note:de,tags.ref:Hermes,tags.description,tags.coffee,tags.tobacco,tags.post_office:brand:wikidata,tags.note,tags.operator:wikidata,tags.ref,tags.currency:EUR,source,tags.diet:gluten_free,tags.leisure,tags.cafe,tags.payment:coins,tags.vending:sweets,tags.vending:toys,tags.service:phone,tags.type,tags.diet:kosher,tags.diet:vegetarian,tags.post_box,tags.operator:wikipedia,tags.changing_table,tags.payment:mastercard,tags.payment:visa,tags.second_hand,tags.toilets:menstrual_products,tags.takeaway,tags.toilets:access,tags.parcel_pickup,tags.drink:coffee,tags.sells:tobacco,tags.name:zh,tags.addr:housename,tags.fixme,tags.contact:mobile,tags.dry_cleaning,tags.ice_cream,tags.drink:beer,tags.tickets:public_transport,tags.diet:vegan,tags.branch,tags.bulk_purchase,tags.diet:organic,tags.frozen_yogurt,tags.reusable_packaging:offer,tags.contact:email,tags.lgbtq,tags.post_office:letter,tags.post_office:parcel_to,tags.addr:place,tags.post_office:parcel_from,tags.post_office:parcel_pickup,tags.ref:deutsche_post,tags.tourism,tags.fair_trade,tags.payment:contactless,tags.payment:maestro,tags.payment:v_pay,tags.payment:app,tags.mapillary,tags.opening_date,tags.contact:facebook,tags.post_office:operator,tags.cash_in,tags.survey:date,tags.post_office:letter_from,tags.post_office:packaging,tags.post_office:stamps,tags.payment:girocard,tags.description:en,tags.old_name,tags.old_addr:housenumber,tags.old_addr:street,tags.reusable_packaging:accept,tags.wikidata,tags.wikipedia,tags.zero_waste,tags.not:brand:wikidata,tags.post_office:post_bank,tags.building,tags.roof:material,tags.roof:shape,tags.vending,tags.layer,tags.disused:shop,tags.service:copy,tags.service:fax,tags.service:scan,tags.self_checkout,tags.payment:american_express,tags.payment:notes,tags.payment:telephone_cards,tags.payment:credit_cards:min_payment,tags.loc_name,tags.delivery,tags.bicycle,tags.craft,tags.indoor:level,tags.name:bg,tags.name:fa,tags.ramp:wheelchair,tags.toilets:charge,tags.smoking:outside,tags.description:name,tags.name:ar,tags.post_office:service_provider:wikidata,tags.wheelchair:description:de,tags.wheelchair:description:en,tags.payment:account_cards,tags.payment:alipay,tags.payment:bancomat,tags.payment:blik,tags.payment:cheque,tags.payment:clipper,tags.payment:cryptocurrencies,tags.payment:diners_club,tags.payment:discover_card,tags.payment:dkv,tags.payment:electronic_purses,tags.payment:ep_easycard,tags.payment:ep_geldkarte,tags.payment:ep_ipass

In [179]:
# Select the 25 columns (fill missing with None if not present)

selected_columns = [
    #"osmid",
    "name", "brand", "operator",
    "addr:street", "addr:housenumber", "addr:postcode", "addr:suburb","addr:city", "addr:country",
    "phone", "email", "website", "opening_hours",
    "payment:visa", "payment:mastercard","payment:girocard", "dispensing", "delivery","surveillance","wheelchair", "building",
    "latitude", "longitude", "geometry",
    # placeholders for enrichment
    #"neighbourhood", "district",
    # add source info
    "source"
]  # Added the closing bracket here

In [180]:
# Rename map for only the columns that need renaming

rename_map = {
    "addr:street": "street",
    "addr:housenumber": "housenumber",
    "addr:postcode": "postcode",
    "addr:suburb": "suburb",
    "addr:city": "city",
    "addr:country": "country",
    "payment:visa": "payment_visa",
    "payment:mastercard": "payment_mastercard",
    "payment:girocard": "payment_girocard",
    "opening_hours": "openinghours",
    "wheelchair": "wheelchair_accessible",
    "building": "building_type"
}

In [182]:
# Preview the final DataFrame
spatis_gdf.head()

,osm_id,osm_type,brand,operator,opening_hours,phone,website,source,district_id,neighborhood_id,neighborhood_name,latitude,longitude,geometry
0,26867411,node,NaN,Bavaria Petrol,Mo-Fr 07:00-22:00; Sa 08:00-22:00,NaN,NaN,NaN,11,11,Deutschland,52.501974,13.294496,POINT (13.2945 52.50197)
1,29997723,node,Aral,Anne Notzke,24/7,NaN,https://tankstelle.aral.de/tankstelle/berlin/m...,NaN,11,11,Deutschland,52.508370,13.280947,POINT (13.28095 52.50837)
2,63253672,node,NaN,NaN,24/7,NaN,NaN,NaN,11,11,Deutschland,52.499322,13.296118,POINT (13.29612 52.49932)
3,253616592,node,NaN,NaN,Mo-Fr 08:00-20:00; Sa 08:00-19:00,NaN,NaN,NaN,11,11,Deutschland,52.511047,13.462698,POINT (13.4627 52.51105)
4,266629404,node,EDEKA,Heinz Vollack,Mo-Fr 07:00-19:00; Sa 07:00-13:00; PH off,+49 30 5677706,NaN,NaN,11,11,Deutschland,52.509209,13.587500,POINT (13.5875 52.50921)


In [183]:
# Data types and non-null counts

spatis_gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1607 entries, 0 to 1606
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   osm_id             1607 non-null   int64   
 1   osm_type           1607 non-null   object  
 2   brand              56 non-null     object  
 3   operator           58 non-null     object  
 4   opening_hours      719 non-null    object  
 5   phone              70 non-null     object  
 6   website            57 non-null     object  
 7   source             47 non-null     object  
 8   district_id        1607 non-null   object  
 9   neighborhood_id    1607 non-null   object  
 10  neighborhood_name  1607 non-null   object  
 11  latitude           1607 non-null   float64 
 12  longitude          1607 non-null   float64 
 13  geometry           1607 non-null   geometry
dtypes: float64(2), geometry(1), int64(1), object(10)
memory usage: 175.9+ KB


In [184]:
import osmnx as ox
import geopandas as gpd

districts_gdf = ox.features_from_place(
    "Berlin, Germany",
    {"boundary": "administrative", "admin_level": "9"}
)

districts_gdf.head()


geometry  \
element  id                                                         
relation 16328  POLYGON ((13.30038 52.56998, 13.3019 52.57144,...   
         16330  POLYGON ((13.26423 52.62686, 13.26438 52.62802...   
         16334  POLYGON ((13.2082 52.59899, 13.20724 52.59987,...   
         16343  POLYGON ((13.10932 52.45071, 13.10956 52.45108...   
         16346  POLYGON ((13.27034 52.54934, 13.27061 52.54934...   

                      boundary                 name  ref admin_level name:ab  \
element  id                                                                    
relation 16328  administrative        Reinickendorf  NaN          10     NaN   
         16330  administrative              Frohnau  NaN          10     NaN   
         16334  administrative        Reinickendorf  NaN           9     NaN   
         16343  administrative              Spandau  NaN           9     NaN   
         16346  administrative  Charlottenburg-Nord  NaN          10     NaN   

               name:af name:als name:am name:an name:ang name:ar name:arc  \
element  id                                                                 
relation 16328     NaN      NaN     NaN     NaN      NaN     NaN      NaN   
         16330     NaN      NaN     NaN     NaN      NaN     NaN      NaN   
         16334     NaN      NaN     NaN     NaN      NaN     NaN      NaN   
         16343     NaN      NaN     NaN     NaN      NaN     NaN      NaN   
         16346     NaN      NaN     NaN     NaN      NaN     NaN      NaN   

               name:arz name:ast name:av name:az name:ba name:bar  \
element  id                                                         
relation 16328      NaN      NaN     NaN     NaN     NaN      NaN   
         16330      NaN      NaN     NaN     NaN     NaN      NaN   
         16334      NaN      NaN     NaN     NaN     NaN      NaN   
         16343      NaN      NaN     NaN     NaN     NaN      NaN   
         16346      NaN      NaN     NaN     NaN     NaN      NaN   

               name:bat-smg name:be name:be-tarask name:bg name:bi name:bn  \
element  id                                                                  
relation 16328          NaN     NaN            NaN     NaN     NaN     NaN   
         16330          NaN     NaN            NaN     NaN     NaN     NaN   
         16334          NaN     NaN            NaN     NaN     NaN     NaN   
         16343          NaN     NaN            NaN     NaN     NaN     NaN   
         16346          NaN     NaN            NaN     NaN     NaN     NaN   

               name:bo name:br name:bs name:bxr name:ca name:cbk-zam name:ce  \
element  id                                                                    
relation 16328     NaN     NaN     NaN      NaN     NaN          NaN     NaN   
         16330     NaN     NaN     NaN      NaN     NaN          NaN     NaN   
         16334     NaN     NaN     NaN      NaN     NaN          NaN     NaN   
         16343     NaN     NaN     NaN      NaN     NaN          NaN     NaN   
         16346     NaN     NaN     NaN      NaN     NaN          NaN     NaN   

               name:ckb name:co name:crh name:cs name:csb name:cu name:cv  \
element  id                                                                 
relation 16328      NaN     NaN      NaN     NaN      NaN     NaN     NaN   
         16330      NaN     NaN      NaN     NaN      NaN     NaN     NaN   
         16334      NaN     NaN      NaN     NaN      NaN     NaN     NaN   
         16343      NaN     NaN      NaN     NaN      NaN     NaN     NaN   
         16346      NaN     NaN      NaN     NaN      NaN     NaN     NaN   

               name:cy name:da        name:de name:diq name:dsb name:el  \
element  id                                                               
relation 16328     NaN     NaN  Reinickendorf      NaN      NaN     NaN   
         16330     NaN     NaN            NaN      NaN      NaN     NaN   
         16334     NaN     NaN          

In [185]:
districts_clean = districts_gdf[['geometry', 'name', 'admin_level']]
districts_clean.head()


geometry  \
element  id                                                         
relation 16328  POLYGON ((13.30038 52.56998, 13.3019 52.57144,...   
         16330  POLYGON ((13.26423 52.62686, 13.26438 52.62802...   
         16334  POLYGON ((13.2082 52.59899, 13.20724 52.59987,...   
         16343  POLYGON ((13.10932 52.45071, 13.10956 52.45108...   
         16346  POLYGON ((13.27034 52.54934, 13.27061 52.54934...   

                               name admin_level  
element  id                                      
relation 16328        Reinickendorf          10  
         16330              Frohnau          10  
         16334        Reinickendorf           9  
         16343              Spandau           9  
         16346  Charlottenburg-Nord          10

In [186]:
districts_clean['geometry'] = districts_clean['geometry'].buffer(0)


/opt/anaconda3/lib/python3.13/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [187]:
import os

# Create the directory structure if it doesn't exist
os.makedirs("spatis/sources", exist_ok=True)

# Now save the file
districts_clean.to_file("spatis/sources/berlin_districts.geojson", driver="GeoJSON")

In [188]:
spatis_gdf = spatis_gdf.to_crs("EPSG:4326")
districts_clean = districts_clean.to_crs("EPSG:4326")


In [189]:
spatis_with_districts = gpd.sjoin(
    spatis_gdf, 
    districts_clean, 
    how="left", 
    predicate="within"
)


In [190]:
spatis_with_districts = spatis_with_districts.rename(
    columns={
        "name_left": "store_name",
        "name_right": "district"
    }
)


In [191]:
spatis_with_districts.to_file("spatis/sources/spatis_with_districts.geojson", driver="GeoJSON")


In [231]:
print(spatis_gdf.columns)



Index(['osm_type', 'osm_id', 'latitude', 'longitude', 'tags.addr:city',
       'tags.addr:country', 'tags.addr:housenumber', 'tags.addr:postcode',
       'tags.addr:street', 'tags.addr:suburb',
       ...
       'tags.post_office:opening_hours', 'tags.money_transfer', 'tags.mobile',
       'tags.fuel:HGV_diesel', 'tags.fuel:octane_100',
       'tags.post_office:id_check', 'tags.name:ko', 'tags.public_transport',
       'tags.ticket', 'geometry'],
      dtype='object', length=248)


In [244]:
import osmnx as ox
import geopandas as gpd
import os

# ensure mapping folder exists
os.makedirs("mapping", exist_ok=True)

districts = ox.features_from_place(
    "Berlin, Germany",
    {"boundary": "administrative", "admin_level": "9"}
).to_crs(4326)

districts.to_file("mapping/districts.geojson", driver="GeoJSON")


In [245]:
districts = gpd.read_file("mapping/districts.geojson").to_crs(4326)

spatis_with_district = gpd.sjoin(
    spatis_gdf,
    districts[['name','geometry']],
    how='left',
    predicate='within'
)

spatis_with_district = spatis_with_district.rename(columns={'name':'district_name'})


In [246]:
spatis_fp = "spatis/spatis_data.geojson"


In [247]:
spatis_with_district_name = gpd.sjoin(
    spatis_gdf,
    districts[['name', 'geometry']],
    how='left',
    predicate='within'
).rename(columns={'name': 'district_name'}).drop(columns=['index_right'], errors='ignore')


In [321]:
import os
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point


mapping_dir = "mapping"
output_dir = "spatis/sources"
os.makedirs(mapping_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)


# 1. Fetch SPATIS from OSM

query = """
[out:json];
(
  node[shop=convenience](52.4,13.2,52.7,13.6);
  node[shop=kiosk](52.4,13.2,52.7,13.6);
);
out body;
"""

url = "https://overpass-api.de/api/interpreter"
resp = requests.post(url, data=query, timeout=180)
resp.raise_for_status()
data = resp.json()
elements = data.get("elements", [])
df = pd.json_normalize(elements)
df = df[df['lat'].notna() & df['lon'].notna()]

spatis_gdf = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df['lon'].astype(float), df['lat'].astype(float))],
    crs="EPSG:4326"
)

# Minimal cleanup / rename to match schema
spatis_gdf = spatis_gdf.rename(columns={
    'tags.name': 'name',
    'tags.brand': 'brand',
    'tags.operator': 'operator',
    'tags.opening_hours': 'opening_hours',
    'tags.phone': 'phone',
    'tags.website': 'website',
    'tags.source': 'source',
    'lat': 'latitude',
    'lon': 'longitude',
    'type': 'osm_type',
    'id': 'osm_id'
})

print("SPATIS fetched with", len(spatis_gdf), "records")


# 2. District mapping

district_mapping = {
    'Mitte': '11001001',
    'Friedrichshain-Kreuzberg': '11002002',
    'Pankow': '11003003',
    'Charlottenburg-Wilmersdorf': '11004004',
    'Spandau': '11005005',
    'Steglitz-Zehlendorf': '11006006',
    'Tempelhof-Schöneberg': '11007007',
    'Neukölln': '11008008',
    'Treptow-Köpenick': '11009009',
    'Marzahn-Hellersdorf': '11010010',
    'Lichtenberg': '11011011',
    'Reinickendorf': '11012012'
}

# Map smaller neighborhoods / suburbs to official districts
neighborhood_to_district = {
    'Kreuzberg': 'Friedrichshain-Kreuzberg',
    'Friedrichshain': 'Friedrichshain-Kreuzberg',
    'Charlottenburg': 'Charlottenburg-Wilmersdorf',
    'Wilmersdorf': 'Charlottenburg-Wilmersdorf',
    'Marzahn': 'Marzahn-Hellersdorf',
    'Hellersdorf': 'Marzahn-Hellersdorf',
    'Schöneberg': 'Tempelhof-Schöneberg',
    'Tempelhof': 'Tempelhof-Schöneberg',
    'Moabit': 'Mitte',
    'Prenzlauer Berg': 'Pankow',
    'Wedding': 'Mitte',
    'Lichtenberg': 'Lichtenberg',
    'Neukölln': 'Neukölln',
    'Treptow': 'Treptow-Köpenick',
    'Köpenick': 'Treptow-Köpenick',
   
}

# Create district_name column from OSM suburb
spatis_gdf['district_name'] = spatis_gdf['tags.addr:suburb'].replace(neighborhood_to_district)
spatis_gdf['district_id'] = spatis_gdf['district_name'].map(district_mapping)

# Check unmapped districts
unmapped = spatis_gdf[spatis_gdf['district_id'].isna()]['district_name'].unique()
if len(unmapped) > 0:
    print("⚠️ Unmapped districts remaining:", unmapped)


# 3. Load neighborhoods (optional)

neighborhoods_fp = os.path.join(mapping_dir, "neighborhoods.geojson")
if os.path.exists(neighborhoods_fp):
    neighborhoods = gpd.read_file(neighborhoods_fp).to_crs(epsg=4326)
    if 'neighborhood_id' not in neighborhoods.columns:
        neighborhoods['neighborhood_id'] = neighborhoods.index.astype(str)
    spatis_gdf = gpd.sjoin(
        spatis_gdf,
        neighborhoods[['neighborhood_id', 'name', 'geometry']],
        how='left',
        predicate='within'
    ).rename(columns={'name': 'neighborhood_name'}).drop(columns=['index_right'], errors='ignore')
else:
    spatis_gdf['neighborhood_id'] = None
    spatis_gdf['neighborhood_name'] = None


# 4. Extract coordinates

spatis_gdf['longitude'] = spatis_gdf.geometry.x
spatis_gdf['latitude'] = spatis_gdf.geometry.y


# 5. Define final columns

final_cols = [
    'osm_id', 'osm_type', 'name', 'brand', 'operator', 'opening_hours', 'phone', 'website', 'source',
    'district_id', 'district_name',
    'neighborhood_id', 'neighborhood_name',
    'longitude', 'latitude', 'geometry'
]
final_cols = [c for c in final_cols if c in spatis_gdf.columns]
spatis_final = spatis_gdf[final_cols].copy()


# 6. Save outputs

geojson_fp = os.path.join(output_dir, "spatis_with_admins.geojson")
csv_fp = os.path.join(output_dir, "spatis_with_admins.csv")

spatis_final.to_file(geojson_fp, driver="GeoJSON")
spatis_final.drop(columns='geometry').to_csv(csv_fp, index=False)

print(f"✅ Enriched SPATIS saved. Records: {len(spatis_final)}")


SPATIS fetched with 1607 records
⚠️ Unmapped districts remaining: ['Halensee' nan 'Kaulsdorf' 'Gesundbrunnen' 'Lichterfelde' 'Friedenau'
 'Baumschulenweg' 'Weißensee' 'Oberschöneweide' 'Rudow' 'Zehlendorf'
 'Lübars' 'Steglitz' 'Haselhorst' 'Siemensstadt' 'Alt-Treptow' 'Lankwitz'
 'Fennpfuhl' 'Westend' 'Tegel' 'Adlershof' 'Schildow' 'Heinersdorf'
 'Niederschönhausen' 'Karlshorst' 'Mühlenbeck' 'Schlachtensee' 'Grünau'
 'Rummelsburg' 'Schwanebeck' 'Wittenau' 'Hakenfelde' 'Mariendorf'
 'Alt-Hohenschönhausen' 'Zepernick' 'Wilhelmsruh' 'Biesdorf']
✅ Enriched SPATIS saved. Records: 6449


In [319]:
import os

# Path to save the enriched CSV
save_path = "/Users/harrisongoodman/Documents/spatis_with_admins.csv"

# Assuming your enriched GeoDataFrame is called spatis_final
# Drop the geometry column for CSV export
spatis_final.drop(columns='geometry').to_csv(save_path, index=False)

print("Enriched CSV saved at:", save_path)


Enriched CSV saved at: /Users/harrisongoodman/Documents/spatis_with_admins.csv
